# 🧠 SLD-VM Pipeline — SVAMP & WinoGrande (FIXED)

## Bugs Fixed vs Original
| # | Bug | Fix |
|---|-----|-----|
| 1 | `extract_svamp_answer` was fully commented out → NameError | Restored and improved |
| 2 | `evaluate()` had extraction line commented out → `pred` was raw text, never parsed | Fixed: output stored, extract_fn called properly |
| 3 | `evaluate()` referenced undefined `output` variable in `records.append()` | Fixed: variable now properly assigned |
| 4 | `generate()` default system_message forced single-word answer → broke CoT/few-shot | Removed bad default; each call passes its own system message |
| 5 | Few-shot prompts injected into chat template → model continued generating examples, not solving | Fixed: few-shot examples embedded as proper user/assistant chat turns |
| 6 | CoT extractor picked first number in reasoning instead of final answer | Fixed: prioritises `Answer: X` pattern, then `#### X`, then last number |
| 7 | `extract_wino_answer` picked LATER mention (wrong heuristic) | Fixed: picks FIRST mention after removing prompt echoes; cleaner regex |
| 8 | WinoGrande zero-shot crashed with `TypeError` (lambda arity mismatch) | Fixed: unified evaluator, no arity mismatch |
| 9 | `evaluate_wino()` defined but never executed | Fixed: single unified `evaluate()` handles both SVAMP and WinoGrande |
| 10 | `generate()` printed inside function (double-printed every output) | Fixed: verbose flag controls printing |
| 11 | SC/VCE cells referenced prompts from broken prior cells | Fixed: best prompt selected cleanly after Phase 1 |

---
## Strategy
| Phase | What we build |
|-------|---------------|
| 0 | Setup & model load |
| 1 | Baselines (zero-shot, few-shot, CoT) |
| 2 | Structured CoT |
| 3 | Self-Consistency (5-sample majority vote) |
| 4 | Rule-based VCE |
| 5 | Results table |

**Model:** `microsoft/Phi-4-mini-instruct`

**Expected accuracy ranges (Phi-4-mini):**
- SVAMP zero-shot: ~55–70%
- WinoGrande zero-shot: ~60–72%


## ⚙️ Phase 0 — Setup

In [1]:
from huggingface_hub import login
login("")
print('✅ HuggingFace login done')

✅ HuggingFace login done


In [2]:
# Uncomment and run once if packages are missing
# !pip install -q -U transformers accelerate datasets

In [3]:
import torch
import json
import re
import random
from collections import Counter
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers

# ── Environment ──────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Transformers: {transformers.__version__}')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Results tracker ───────────────────────────────────────────────────────────
RESULTS = {'SVAMP': {}, 'WinoGrande': {}}
print('\n✅ Imports done. Results tracker initialised.')

Device: cuda
GPU : Tesla P100-PCIE-16GB
VRAM: 17.1 GB
Transformers: 5.2.0

✅ Imports done. Results tracker initialised.


In [4]:
MODEL_ID = 'microsoft/Phi-4-mini-instruct'
print(f'Loading {MODEL_ID} ...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
model.eval()
print('✅ Model loaded')

Loading microsoft/Phi-4-mini-instruct ...


config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

✅ Model loaded


In [5]:
# ── Inference helpers ─────────────────────────────────────────────────────────
# FIX #4: No bad default system message. Each caller provides what is needed.
# FIX #10: verbose=False by default so evaluate() is clean.

def build_prompt(user_message: str, system_message: str = None) -> str:
    """Apply Phi-4 chat template."""
    messages = []
    if system_message:
        messages.append({'role': 'system', 'content': system_message})
    messages.append({'role': 'user', 'content': user_message})
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def generate(
    user_message: str,
    max_new_tokens: int = 256,
    temperature: float = 0.0,
    do_sample: bool = False,
    system_message: str = None,
    verbose: bool = False
) -> str:
    """Run inference; return only newly generated text."""
    prompt = build_prompt(user_message, system_message)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            pad_token_id=tokenizer.eos_token_id
        )

    result = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()
    if verbose:
        print(result)
    return result


def generate_n(
    user_message: str,
    n: int = 5,
    temperature: float = 0.7,
    max_new_tokens: int = 256,
    system_message: str = None
) -> list:
    """Generate n diverse responses for self-consistency."""
    return [
        generate(user_message, max_new_tokens=max_new_tokens,
                 temperature=temperature, do_sample=True,
                 system_message=system_message)
        for _ in range(n)
    ]


# Sanity check
test_out = generate('What is 3 + 5?', system_message='Answer with only the number.')
print(f'Sanity check: 3 + 5 = {test_out!r}  (expected: 8)')
print('✅ Inference functions ready')

Sanity check: 3 + 5 = '8'  (expected: 8)
✅ Inference functions ready


## 📊 Phase 1 — Baselines

In [6]:
N_SAMPLES = 100

# ── SVAMP ────────────────────────────────────────────────────────────────────
print('Loading SVAMP ...')
svamp_raw = load_dataset('ChilleD/SVAMP', split='train')
svamp_samples = svamp_raw.shuffle(seed=SEED).select(range(N_SAMPLES))
print(f'  Loaded {len(svamp_samples)} samples')
print(f'  Keys: {list(svamp_samples[0].keys())}')

# ── WinoGrande ───────────────────────────────────────────────────────────────
print('\nLoading WinoGrande ...')
wino_raw = load_dataset('allenai/winogrande', 'winogrande_xl', split='validation')
wino_samples = wino_raw.shuffle(seed=SEED).select(range(N_SAMPLES))
print(f'  Loaded {len(wino_samples)} samples')
print(f'  Keys: {list(wino_samples[0].keys())}')
print(f'  Sample: {wino_samples[0]}')
print('\n✅ Datasets ready')

Loading SVAMP ...


README.md:   0%|          | 0.00/675 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/111k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/54.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/700 [00:00<?, ? examples/s]

...

winogrande_xl/validation-00000-of-00001.(…):   0%|          | 0.00/85.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40398 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1767 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1267 [00:00<?, ? examples/s]

  Loaded 100 samples
  Keys: ['sentence', 'option1', 'option2', 'answer']
  Sample: {'sentence': 'Patricia decided to buy Felicia dinner because they had been through a lot and _ just inherited some money.', 'option1': 'Patricia', 'option2': 'Felicia', 'answer': '1'}

✅ Datasets ready


In [7]:
# ── Answer extractors ─────────────────────────────────────────────────────────
# FIX #1: extract_svamp_answer was entirely commented out. Restored + improved.
# FIX #6: Now correctly prioritises Answer: X pattern over first number in text.

def extract_svamp_answer(text):
    """
    Extract numeric answer from model output.
    Priority: 'Answer: X' > '#### X' > boxed{X} > last number in text.
    Returns float or None.
    """
    # Already numeric (gold answer field)
    if isinstance(text, (int, float)):
        return float(text)
    if not isinstance(text, str):
        return None

    text_clean = text.replace(',', '')

    # Pattern 1: explicit answer marker (most reliable — comes from CoT reasoning)
    m = re.search(
        r'(?:answer(?:\s+is)?|final\s+answer|result)[:\s]+(-?\d+(?:\.\d+)?)',
        text_clean, re.IGNORECASE
    )
    if m:
        return float(m.group(1))

    # Pattern 2: #### X  (GSM8K / structured CoT style)
    m = re.search(r'####\s*(-?\d+(?:\.\d+)?)', text_clean)
    if m:
        return float(m.group(1))

    # Pattern 3: \boxed{X}
    m = re.search(r'\\boxed\{(-?\d+(?:\.\d+)?)\}', text_clean)
    if m:
        return float(m.group(1))

    # Pattern 4: = X at end of expression
    m = re.search(r'=\s*(-?\d+(?:\.\d+)?)\s*$', text_clean)
    if m:
        return float(m.group(1))

    # Pattern 5: last number in the whole output (least reliable)
    nums = re.findall(r'-?\d+(?:\.\d+)?', text_clean)
    return float(nums[-1]) if nums else None


# FIX #7: Improved WinoGrande extractor.
# Original logic returned LATER mention, which is actually correct for CoT style,
# but broke on echoed prompts. Now strips prompt echo first, then finds final answer.
def extract_wino_answer(text: str, option1: str, option2: str):
    """
    Returns 1 or 2 (matching dataset answer field), or None.
    Strategy:
      1. Look for explicit 'Answer: <word>' or 'answer is <word>'
      2. Look for '1' or '2' label explicitly called out
      3. Count which option appears more in reasoning conclusion
    """
    o1 = option1.lower().strip()
    o2 = option2.lower().strip()
    t  = text.lower().strip()

    # Strategy 1: explicit answer label
    m = re.search(r'(?:answer(?:\s+is)?|choose|chose|option)[:\s]+(.{1,40})', t, re.IGNORECASE)
    if m:
        snippet = m.group(1).strip()
        found1 = o1 in snippet
        found2 = o2 in snippet
        if found1 and not found2:
            return 1
        if found2 and not found1:
            return 2
        # Both or neither in snippet — fall through
        if '1' in snippet and '2' not in snippet:
            return 1
        if '2' in snippet and '1' not in snippet:
            return 2

    # Strategy 2: explicit digit label
    m = re.search(r'\boption\s*([12])\b', t)
    if m:
        return int(m.group(1))

    # Strategy 3: direct word mention
    pos1 = t.rfind(o1)   # last occurrence (conclusion more likely near end)
    pos2 = t.rfind(o2)

    if pos1 == -1 and pos2 == -1:
        # Fallback: first bare 1 or 2
        m = re.search(r'\b([12])\b', t)
        return int(m.group(1)) if m else None
    if pos1 == -1:
        return 2
    if pos2 == -1:
        return 1
    # Whichever appears LAST in text is the concluded answer
    return 1 if pos1 > pos2 else 2


# ── Correctness helpers ───────────────────────────────────────────────────────
def svamp_correct(pred, gold):
    if pred is None:
        return False
    try:
        return abs(float(pred) - float(gold)) < 0.01
    except:
        return False


def wino_correct(pred, gold):
    """gold is '1' or '2' (string). pred is int 1 or 2."""
    if pred is None:
        return False
    return str(pred) == str(gold)


# ── Gold label helpers ────────────────────────────────────────────────────────
def svamp_gold(sample):
    ans = sample.get('Answer', sample.get('answer', 0))
    return float(str(ans).replace(',', ''))


def wino_gold(sample):
    return sample['answer']  # '1' or '2'


print('✅ Extractors ready')
# Quick self-test
assert extract_svamp_answer('The answer is 21') == 21.0
assert extract_svamp_answer('so total = 50 - 29 = 21.') == 21.0
assert extract_svamp_answer('21 pounds') == 21.0
assert extract_svamp_answer('22 days') == 22.0
print('  SVAMP extractor self-tests passed')
assert extract_wino_answer('The answer is Patricia', 'Patricia', 'Felicia') == 1
assert extract_wino_answer('I choose Felicia', 'Patricia', 'Felicia') == 2
print('  WinoGrande extractor self-tests passed')

✅ Extractors ready
  SVAMP extractor self-tests passed
  WinoGrande extractor self-tests passed


In [8]:
# ── Unified evaluation runner ─────────────────────────────────────────────────
# FIX #2: output now stored; extract_fn called correctly.
# FIX #3: 'output' variable no longer referenced before assignment.
# FIX #8: No arity mismatch — extract_fn always receives (output, sample).
#          For SVAMP, wrap extract_svamp_answer: lambda out, s: extract_svamp_answer(out)

def evaluate(
    samples,
    prompt_fn,
    extract_fn,        # signature: (output_text, sample) -> answer
    correct_fn,
    gold_fn,
    label: str,
    system_message: str = None,
    print_first_n: int = 5,
    max_new_tokens: int = 256,
    temperature: float = 0.0
):
    correct = 0
    null_count = 0
    records = []

    for i, sample in enumerate(tqdm(samples, desc=label)):
        prompt = prompt_fn(sample)
        # FIX #2/#3: store output properly, then extract
        output = generate(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            system_message=system_message
        )
        pred = extract_fn(output, sample)   # FIX #8: always 2-arg
        gold = gold_fn(sample)
        is_correct = correct_fn(pred, gold)

        if pred is None:
            null_count += 1
        if is_correct:
            correct += 1

        records.append({
            'output': output,  # FIX #3: now defined
            'pred': pred,
            'gold': gold,
            'correct': is_correct,
            'sample': sample
        })

        if i < print_first_n:
            print(f'\n─── Example {i+1} {"─"*40}')
            print(f'  OUTPUT : {output[:200]}')
            print(f'  PRED   : {pred}  |  GOLD: {gold}  |  CORRECT: {is_correct}')

    acc = correct / len(samples)
    print(f'\n[{label}] Accuracy: {acc:.1%} | Null: {null_count}/{len(samples)}')
    return acc, records


print('✅ Evaluation runner ready')

✅ Evaluation runner ready


In [13]:
# ── SVAMP system message ──────────────────────────────────────────────────────
# FIX #4: Appropriate system message for math — allows reasoning, requires final answer.
SVAMP_SYS = (
    'You are a math problem solver. '
    'Show your working step by step, then write the final numeric answer '
    'on its own line starting with "Answer: "'
)

# ── SVAMP: extractor wrapper (2-arg) ─────────────────────────────────────────
svamp_extract = lambda out, s: extract_svamp_answer(out)

# ── Prompt builders ───────────────────────────────────────────────────────────
def svamp_body_q(sample):
    body = sample.get('Body', sample.get('body', ''))
    q    = sample.get('Question', sample.get('question', ''))
    return f'{body} {q}'.strip()


def svamp_zeroshot_prompt(sample):
    return (
        f'Solve the following math word problem.\n\n'
        f'{svamp_body_q(sample)}\n\n'
        f'Show your reasoning step by step, then state the final numeric answer '
        f'on a line starting with "Answer: "'
    )


# FIX #5: Few-shot via proper chat turns, NOT raw text continuation.
# This stops the model from generating more example problems.
SVAMP_FS_EXAMPLES = [
    {
        'user': 'Solve: There are 5 red balls and 3 blue balls. How many balls in total?\nAnswer on a line starting with "Answer: "',
        'assistant': 'Red balls = 5, blue balls = 3.\nTotal = 5 + 3 = 8\nAnswer: 8'
    },
    {
        'user': 'Solve: A shop had 50 apples. Sold 17 on Monday and 12 on Tuesday. How many left?\nAnswer on a line starting with "Answer: "',
        'assistant': 'Sold = 17 + 12 = 29. Remaining = 50 − 29 = 21\nAnswer: 21'
    },
    {
        'user': 'Solve: Each child needs 3 pencils. There are 8 children. How many pencils needed?\nAnswer on a line starting with "Answer: "',
        'assistant': 'Pencils = 3 × 8 = 24\nAnswer: 24'
    },
]

def svamp_fewshot_prompt(sample):
    """FIX #5: few-shot examples as chat history, not raw text continuation."""
    messages = [{'role': 'system', 'content': SVAMP_SYS}]
    for ex in SVAMP_FS_EXAMPLES:
        messages.append({'role': 'user',      'content': ex['user']})
        messages.append({'role': 'assistant', 'content': ex['assistant']})
    messages.append({
        'role': 'user',
        'content': f'Solve: {svamp_body_q(sample)}\nAnswer on a line starting with "Answer: "'
    })
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

SVAMP_COT_EXAMPLES = [
    {
        'user': (
            'Solve step by step:\n'
            'Danny had 56 bottle caps. He threw away 17 and found 34 more at the park. '
            'How many bottle caps does he have now?\n'
            'End with "Answer: <number>"'
        ),
        'assistant': (
            'Start: 56\n'
            'Threw away: 56 - 17 = 39\n'
            'Found more: 39 + 34 = 73\n'
            'Answer: 73'
        )
    },
    {
        'user': (
            'Solve step by step:\n'
            'A shop had 50 apples. It sold 17 on Monday and 12 on Tuesday. '
            'How many apples are left?\n'
            'End with "Answer: <number>"'
        ),
        'assistant': (
            'Total sold: 17 + 12 = 29\n'
            'Remaining: 50 - 29 = 21\n'
            'Answer: 21'
        )
    },
    {
        'user': (
            'Solve step by step:\n'
            'Each child needs 3 pencils. There are 8 children in the class. '
            'How many pencils are needed in total?\n'
            'End with "Answer: <number>"'
        ),
        'assistant': (
            'Pencils per child: 3\n'
            'Number of children: 8\n'
            'Total: 3 × 8 = 24\n'
            'Answer: 24'
        )
    },
]

def svamp_cot_prompt(sample):
    messages = [{'role': 'system', 'content': SVAMP_SYS}]
    for ex in SVAMP_COT_EXAMPLES:
        messages.append({'role': 'user',      'content': ex['user']})
        messages.append({'role': 'assistant', 'content': ex['assistant']})
    messages.append({
        'role': 'user',
        'content': (
            f'Solve step by step:\n'
            f'{svamp_body_q(sample)}\n'
            f'End with "Answer: <number>"'
        )
    })
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

# Verify sample format
print('Sample SVAMP zero-shot prompt (first 300 chars):')
print(svamp_zeroshot_prompt(svamp_samples[0])[:300])
print('\n✅ SVAMP prompt builders ready')

Sample SVAMP zero-shot prompt (first 300 chars):
Solve the following math word problem.

Danny collects bottle caps. He found 63 bottle caps at the park while he threw away 51 old ones. Now he has 33 bottle caps in his collection. How many bottle caps did danny have at first?

Show your reasoning step by step, then state the final numeric answer o

✅ SVAMP prompt builders ready


In [10]:
# ── SVAMP Baseline: Zero-Shot ─────────────────────────────────────────────────
acc, svamp_zs_records = evaluate(
    svamp_samples,
    prompt_fn=svamp_zeroshot_prompt,
    extract_fn=svamp_extract,
    correct_fn=svamp_correct,
    gold_fn=svamp_gold,
    label='SVAMP Zero-Shot',
    system_message=SVAMP_SYS,
)
RESULTS['SVAMP']['zero_shot'] = acc

SVAMP Zero-Shot:   1%|          | 1/100 [00:06<11:29,  6.97s/it]


─── Example 1 ────────────────────────────────────────
  OUTPUT : Let's denote the number of bottle caps Danny had at first as X.

According to the problem, Danny found 63 bottle caps at the park and threw away 51 old ones. After these transactions, he has 33 bottle
  PRED   : 21.0  |  GOLD: 21.0  |  CORRECT: True


SVAMP Zero-Shot:   2%|▏         | 2/100 [00:11<08:53,  5.44s/it]


─── Example 2 ────────────────────────────────────────
  OUTPUT : To find out how many more pages of math homework Rachel had than biology homework, we need to compare the number of pages for each subject.

Rachel had 11 pages of math homework and 3 pages of biology
  PRED   : 8.0  |  GOLD: 8.0  |  CORRECT: True


SVAMP Zero-Shot:   3%|▎         | 3/100 [00:22<12:47,  7.91s/it]

...


─── Example 4 ────────────────────────────────────────
  OUTPUT : Jerry originally had 4 action figures on the shelf. After adding more, he now has a total of 8 action figures.

To find out how many action figures Jerry added, we subtract the original number of acti
  PRED   : 4.0  |  GOLD: 4.0  |  CORRECT: True


SVAMP Zero-Shot:   5%|▌         | 5/100 [00:30<09:10,  5.79s/it]


─── Example 5 ────────────────────────────────────────
  OUTPUT : To find out how many books Frank read, we need to divide the total number of days he spent reading by the number of days it took him to finish each book.

Total days spent reading: 492 days
Days to fi
  PRED   : 41.0  |  GOLD: 41.0  |  CORRECT: True


SVAMP Zero-Shot: 100%|██████████| 100/100 [09:23<00:00,  5.64s/it]


[SVAMP Zero-Shot] Accuracy: 90.0% | Null: 0/100


In [11]:
# ── SVAMP Baseline: Few-Shot ──────────────────────────────────────────────────
# FIX #5: few-shot prompt already encodes examples as chat turns internally;
# system_message=None because SVAMP_SYS already embedded inside prompt.
acc, svamp_fs_records = evaluate(
    svamp_samples,
    prompt_fn=svamp_fewshot_prompt,
    extract_fn=svamp_extract,
    correct_fn=svamp_correct,
    gold_fn=svamp_gold,
    label='SVAMP Few-Shot',
    system_message=None,   # already embedded in svamp_fewshot_prompt()
)
RESULTS['SVAMP']['few_shot'] = acc

SVAMP Few-Shot:   1%|          | 1/100 [00:03<04:58,  3.02s/it]


─── Example 1 ────────────────────────────────────────
  OUTPUT : Let x be the number of bottle caps Danny had at first.

x - 51 (threw away) + 63 (found) = 33 (now has)

x + 12 = 33

x = 33 - 12

x = 21

Answer: 21
  PRED   : 21.0  |  GOLD: 21.0  |  CORRECT: True


SVAMP Few-Shot:   2%|▏         | 2/100 [00:04<03:36,  2.21s/it]


─── Example 2 ────────────────────────────────────────
  OUTPUT : Math homework = 11 pages, Biology homework = 3 pages.
Difference = 11 - 3 = 8
Answer: 8
  PRED   : 8.0  |  GOLD: 8.0  |  CORRECT: True


SVAMP Few-Shot:   3%|▎         | 3/100 [00:06<02:59,  1.85s/it]

...


─── Example 4 ────────────────────────────────────────
  OUTPUT : Jerry had 4 action figures initially. Now there are 8 action figures. So, he added 8 - 4 = 4 action figures to the shelf.
Answer: 4
  PRED   : 4.0  |  GOLD: 4.0  |  CORRECT: True


SVAMP Few-Shot:   5%|▌         | 5/100 [00:10<03:10,  2.01s/it]


─── Example 5 ────────────────────────────────────────
  OUTPUT : Pages per book = 66, Days per book = 12, Total days = 492

Books = Total days / Days per book = 492 / 12 = 41

Answer: 41
  PRED   : 41.0  |  GOLD: 41.0  |  CORRECT: True


SVAMP Few-Shot: 100%|██████████| 100/100 [02:58<00:00,  1.78s/it]


[SVAMP Few-Shot] Accuracy: 90.0% | Null: 0/100


In [14]:
# ── SVAMP Baseline: CoT ───────────────────────────────────────────────────────
acc, svamp_cot_records = evaluate(
    svamp_samples,
    prompt_fn=svamp_cot_prompt,
    extract_fn=svamp_extract,
    correct_fn=svamp_correct,
    gold_fn=svamp_gold,
    label='SVAMP CoT',
    system_message=SVAMP_SYS,
)
RESULTS['SVAMP']['cot'] = acc

print('\n── SVAMP Baselines ────────────────────────────────')
for k, v in RESULTS['SVAMP'].items():
    print(f'  {k:20s}: {v:.1%}')

SVAMP CoT:   1%|          | 1/100 [00:02<03:35,  2.18s/it]


─── Example 1 ────────────────────────────────────────
  OUTPUT : Current collection: 33
Found at the park: 63
Threw away: 51
Initial collection: 33 - 63 + 51 = 21
Answer: 21
  PRED   : 21.0  |  GOLD: 21.0  |  CORRECT: True


SVAMP CoT:   2%|▏         | 2/100 [00:03<03:11,  1.95s/it]


─── Example 2 ────────────────────────────────────────
  OUTPUT : Pages of math homework: 11
Pages of biology homework: 3
Difference: 11 - 3 = 8
Answer: 8
  PRED   : 8.0  |  GOLD: 8.0  |  CORRECT: True


SVAMP CoT:   3%|▎         | 3/100 [00:05<02:54,  1.79s/it]

...


─── Example 4 ────────────────────────────────────────
  OUTPUT : Initial action figures: 4
Total action figures now: 8
Added action figures: 8 - 4 = 4
Answer: 4
  PRED   : 4.0  |  GOLD: 4.0  |  CORRECT: True


SVAMP CoT:   5%|▌         | 5/100 [00:09<02:57,  1.87s/it]


─── Example 5 ────────────────────────────────────────
  OUTPUT : Pages per book: 66
Days per book: 12
Total days: 492
Books: 492 ÷ 12 = 41
Answer: 41
  PRED   : 41.0  |  GOLD: 41.0  |  CORRECT: True


SVAMP CoT: 100%|██████████| 100/100 [03:42<00:00,  2.23s/it]


[SVAMP CoT] Accuracy: 89.0% | Null: 0/100

── SVAMP Baselines ────────────────────────────────
  zero_shot           : 90.0%
  few_shot            : 90.0%
  cot                 : 89.0%


In [15]:
# ── WinoGrande system message ─────────────────────────────────────────────────
WINO_SYS = (
    'You are a language understanding assistant. '
    'Choose the word that best fills the blank. '
    'Reason briefly, then state your final answer on a line starting with "Answer: "'
)

# ── WinoGrande: extractor wrapper (2-arg) ────────────────────────────────────
def wino_extract(out, sample):
    return extract_wino_answer(out, sample['option1'], sample['option2'])

# ── Few-Shot ──────────────────────────────────────────────────────────────────
WINO_FS_EXAMPLES = [
    {
        'user': (
            'Fill in the blank:\n'
            'Sentence: The trophy doesn\'t fit in the suitcase because _ is too large.\n'
            'Option 1: trophy\n'
            'Option 2: suitcase\n'
            'Answer with "Answer: <word>"'
        ),
        'assistant': 'Answer: trophy'
    },
    {
        'user': (
            'Fill in the blank:\n'
            'Sentence: Sam tried to paint a picture of sheep, but they came out looking more like _.\n'
            'Option 1: dogs\n'
            'Option 2: sheep\n'
            'Answer with "Answer: <word>"'
        ),
        'assistant': 'Answer: dogs'
    },
    {
        'user': (
            'Fill in the blank:\n'
            'Sentence: The surgeon performed the operation on the patient even though _ was very tired.\n'
            'Option 1: the surgeon\n'
            'Option 2: the patient\n'
            'Answer with "Answer: <word>"'
        ),
        'assistant': 'Answer: the surgeon'
    },
]

def wino_fewshot_prompt(sample):
    messages = [{'role': 'system', 'content': WINO_SYS}]
    for ex in WINO_FS_EXAMPLES:
        messages.append({'role': 'user',      'content': ex['user']})
        messages.append({'role': 'assistant', 'content': ex['assistant']})
    messages.append({
        'role': 'user',
        'content': (
            f'Fill in the blank:\n'
            f'Sentence: {sample["sentence"]}\n'
            f'Option 1: {sample["option1"]}\n'
            f'Option 2: {sample["option2"]}\n'
            f'Answer with "Answer: <word>"'
        )
    })
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# ── Chain-of-Thought ──────────────────────────────────────────────────────────
WINO_COT_EXAMPLES = [
    {
        'user': (
            'Fill in the blank:\n'
            'Sentence: The trophy doesn\'t fit in the suitcase because _ is too large.\n'
            'Option 1: trophy\n'
            'Option 2: suitcase\n'
            'Reason step by step, then write "Answer: <word>"'
        ),
        'assistant': (
            'What does the blank refer to? The cause of not fitting.\n'
            'Which option causes not fitting? The thing that is too large.\n'
            'A trophy cannot fit inside a suitcase if the trophy is too large.\n'
            'Answer: trophy'
        )
    },
    {
        'user': (
            'Fill in the blank:\n'
            'Sentence: Sam tried to paint a picture of sheep, but they came out looking more like _.\n'
            'Option 1: dogs\n'
            'Option 2: sheep\n'
            'Reason step by step, then write "Answer: <word>"'
        ),
        'assistant': (
            'What does the blank refer to? What the painting accidentally resembled.\n'
            'Sam was trying to paint sheep but failed.\n'
            'The result must look like something other than sheep — dogs.\n'
            'Answer: dogs'
        )
    },
    {
        'user': (
            'Fill in the blank:\n'
            'Sentence: The surgeon performed the operation on the patient even though _ was very tired.\n'
            'Option 1: the surgeon\n'
            'Option 2: the patient\n'
            'Reason step by step, then write "Answer: <word>"'
        ),
        'assistant': (
            'What does the blank refer to? Who was tired despite the operation happening.\n'
            'The patient being tired would not stop a surgery.\n'
            'The surgeon being tired is the meaningful constraint — yet they operated anyway.\n'
            'Answer: the surgeon'
        )
    },
]

def wino_cot_prompt(sample):
    messages = [{'role': 'system', 'content': WINO_SYS}]
    for ex in WINO_COT_EXAMPLES:
        messages.append({'role': 'user',      'content': ex['user']})
        messages.append({'role': 'assistant', 'content': ex['assistant']})
    messages.append({
        'role': 'user',
        'content': (
            f'Fill in the blank:\n'
            f'Sentence: {sample["sentence"]}\n'
            f'Option 1: {sample["option1"]}\n'
            f'Option 2: {sample["option2"]}\n'
            f'Reason step by step, then write "Answer: <word>"'
        )
    })
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# ── Zero-Shot ─────────────────────────────────────────────────────────────────
def wino_zeroshot_prompt(sample):
    return (
        f'Fill in the blank with the correct option.\n\n'
        f'Sentence: {sample["sentence"]}\n'
        f'Option 1: {sample["option1"]}\n'
        f'Option 2: {sample["option2"]}\n\n'
        f'Write "Answer: <word>" on its own line.'
    )

# Verify format
print('WinoGrande sample check:')
s = wino_samples[0]
print(f'  Sentence : {s["sentence"]}')
print(f'  Option 1 : {s["option1"]}')
print(f'  Option 2 : {s["option2"]}')
print(f'  Answer   : {s["answer"]}')
print('\n✅ WinoGrande prompt builders ready')

WinoGrande sample check:
  Sentence : Patricia decided to buy Felicia dinner because they had been through a lot and _ just inherited some money.
  Option 1 : Patricia
  Option 2 : Felicia
  Answer   : 1

✅ WinoGrande prompt builders ready


In [16]:
# ── WinoGrande Baseline: Zero-Shot ───────────────────────────────────────────
# FIX #8/#9: now uses the same unified evaluate() — no arity mismatch
acc, wino_zs_records = evaluate(
    wino_samples,
    prompt_fn=wino_zeroshot_prompt,
    extract_fn=wino_extract,
    correct_fn=wino_correct,
    gold_fn=wino_gold,
    label='WinoGrande Zero-Shot',
    system_message=WINO_SYS,
)
RESULTS['WinoGrande']['zero_shot'] = acc

WinoGrande Zero-Shot:   1%|          | 1/100 [00:00<00:31,  3.16it/s]


─── Example 1 ────────────────────────────────────────
  OUTPUT : Answer: Felicia
  PRED   : 2  |  GOLD: 1  |  CORRECT: False


WinoGrande Zero-Shot:   2%|▏         | 2/100 [00:00<00:27,  3.52it/s]


─── Example 2 ────────────────────────────────────────
  OUTPUT : Answer: north
  PRED   : 2  |  GOLD: 2  |  CORRECT: True


WinoGrande Zero-Shot:   3%|▎         | 3/100 [00:02<01:25,  1.14it/s]

...


─── Example 4 ────────────────────────────────────────
  OUTPUT : Reason: The sentence implies that one of the two has more money to spend, which would make it easier to follow a budget. Since the diner is being compared favorably to the food truck, it suggests that
  PRED   : 1  |  GOLD: 1  |  CORRECT: True


WinoGrande Zero-Shot:   5%|▌         | 5/100 [00:05<02:17,  1.45s/it]


─── Example 5 ────────────────────────────────────────
  OUTPUT : Reason: The headphone is closer to John's ears, making it more likely for him to hear it over the alarm clock.

Answer: headphone
  PRED   : 1  |  GOLD: 1  |  CORRECT: True


WinoGrande Zero-Shot: 100%|██████████| 100/100 [02:07<00:00,  1.27s/it]


[WinoGrande Zero-Shot] Accuracy: 68.0% | Null: 0/100


In [17]:
# ── WinoGrande Baseline: Few-Shot ────────────────────────────────────────────
acc, wino_fs_records = evaluate(
    wino_samples,
    prompt_fn=wino_fewshot_prompt,
    extract_fn=wino_extract,
    correct_fn=wino_correct,
    gold_fn=wino_gold,
    label='WinoGrande Few-Shot',
    system_message=None,   # embedded inside prompt
)
RESULTS['WinoGrande']['few_shot'] = acc

WinoGrande Few-Shot:   1%|          | 1/100 [00:00<00:46,  2.14it/s]


─── Example 1 ────────────────────────────────────────
  OUTPUT : Answer: Felicia
  PRED   : 2  |  GOLD: 1  |  CORRECT: False


WinoGrande Few-Shot:   2%|▏         | 2/100 [00:00<00:43,  2.27it/s]


─── Example 2 ────────────────────────────────────────
  OUTPUT : Answer: north
  PRED   : 2  |  GOLD: 2  |  CORRECT: True


WinoGrande Few-Shot:   3%|▎         | 3/100 [00:01<00:42,  2.31it/s]

...


─── Example 4 ────────────────────────────────────────
  OUTPUT : Answer: diner
  PRED   : 1  |  GOLD: 1  |  CORRECT: True


WinoGrande Few-Shot:   5%|▌         | 5/100 [00:02<00:40,  2.34it/s]


─── Example 5 ────────────────────────────────────────
  OUTPUT : Answer: headphone
  PRED   : 1  |  GOLD: 1  |  CORRECT: True


WinoGrande Few-Shot: 100%|██████████| 100/100 [00:42<00:00,  2.34it/s]


[WinoGrande Few-Shot] Accuracy: 67.0% | Null: 0/100


In [18]:
# ── WinoGrande Baseline: CoT ─────────────────────────────────────────────────
acc, wino_cot_records = evaluate(
    wino_samples,
    prompt_fn=wino_cot_prompt,
    extract_fn=wino_extract,
    correct_fn=wino_correct,
    gold_fn=wino_gold,
    label='WinoGrande CoT',
    system_message=WINO_SYS,
)
RESULTS['WinoGrande']['cot'] = acc

print('\n── WinoGrande Baselines ───────────────────────────')
for k, v in RESULTS['WinoGrande'].items():
    print(f'  {k:20s}: {v:.1%}')
print('  (random baseline = 50.0%)')

print('\n── SVAMP Baselines ────────────────────────────────')
for k, v in RESULTS['SVAMP'].items():
    print(f'  {k:20s}: {v:.1%}')

WinoGrande CoT:   1%|          | 1/100 [00:02<04:50,  2.93s/it]


─── Example 1 ────────────────────────────────────────
  OUTPUT : What does the blank refer to? Who just inherited money.
Patricia is the one buying dinner, implying she is not the one with the new money.
Felicia is the one being treated to dinner, suggesting she is
  PRED   : 2  |  GOLD: 1  |  CORRECT: False


WinoGrande CoT:   2%|▏         | 2/100 [00:05<04:30,  2.76s/it]


─── Example 2 ────────────────────────────────────────
  OUTPUT : What does the blank refer to? Where there was more snow.
The sentence states that the north had warmer clothing due to more snow.
Therefore, the blank should refer to the place with more snow, which i
  PRED   : 2  |  GOLD: 2  |  CORRECT: True


WinoGrande CoT:   3%|▎         | 3/100 [00:08<04:33,  2.82s/it]

...


─── Example 4 ────────────────────────────────────────
  OUTPUT : What does the blank refer to? Whose budget was easier to follow.
The diner had an easier time following their budget, implying they had more money to spend.
The food truck would have a harder time fol
  PRED   : 1  |  GOLD: 1  |  CORRECT: True


WinoGrande CoT:   5%|▌         | 5/100 [00:14<04:27,  2.81s/it]


─── Example 5 ────────────────────────────────────────
  OUTPUT : What does the blank refer to? What is interfering with John's ability to hear.
The headphone is on his head, which would block out sound.
The clock is not on his head, so it wouldn't be the reason for
  PRED   : 1  |  GOLD: 1  |  CORRECT: True


WinoGrande CoT: 100%|██████████| 100/100 [04:43<00:00,  2.84s/it]


[WinoGrande CoT] Accuracy: 70.0% | Null: 0/100

── WinoGrande Baselines ───────────────────────────
  zero_shot           : 68.0%
  few_shot            : 67.0%
  cot                 : 70.0%
  (random baseline = 50.0%)

── SVAMP Baselines ────────────────────────────────
  zero_shot           : 90.0%
  few_shot            : 90.0%
  cot                 : 89.0%


## 🔧 Phase 2 — Structured CoT

In [19]:
# ── Structured CoT: SVAMP ────────────────────────────────────────────────────
# FIX #5: few-shot examples in chat turns, not raw text.
SVAMP_STRUCT_EXAMPLES = [
    {
        'user': 'Solve step by step:\nThere are 5 red balls and 3 blue balls. How many balls in total?\nEnd with "Answer: <number>"',
        'assistant': 'Step 1 (numbers): red=5, blue=3\nStep 2 (operation): 5 + 3 = 8\nAnswer: 8'
    },
    {
        'user': 'Solve step by step:\nA shop had 50 apples. Sold 17 Monday and 12 Tuesday. How many left?\nEnd with "Answer: <number>"',
        'assistant': 'Step 1 (numbers): start=50, mon=17, tue=12\nStep 2 (operation): sold = 17 + 12 = 29\nStep 3 (operation): left = 50 - 29 = 21\nAnswer: 21'
    },
]

def svamp_structured_cot_prompt(sample):
    messages = [{'role': 'system', 'content': SVAMP_SYS}]
    for ex in SVAMP_STRUCT_EXAMPLES:
        messages.append({'role': 'user',      'content': ex['user']})
        messages.append({'role': 'assistant', 'content': ex['assistant']})
    messages.append({
        'role': 'user',
        'content': f'Solve step by step:\n{svamp_body_q(sample)}\nEnd with "Answer: <number>"'
    })
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


acc, svamp_scot_records = evaluate(
    svamp_samples,
    prompt_fn=svamp_structured_cot_prompt,
    extract_fn=svamp_extract,
    correct_fn=svamp_correct,
    gold_fn=svamp_gold,
    label='SVAMP Structured CoT',
    system_message=None,
)
RESULTS['SVAMP']['structured_cot'] = acc

SVAMP Structured CoT:   1%|          | 1/100 [00:02<03:50,  2.33s/it]


─── Example 1 ────────────────────────────────────────
  OUTPUT : Step 1 (numbers): found=63, thrown=51, now=33
Step 2 (operation): total before = now + thrown = 33 + 51 = 84
Answer: 84
  PRED   : 84.0  |  GOLD: 21.0  |  CORRECT: False


SVAMP Structured CoT:   2%|▏         | 2/100 [00:04<03:20,  2.05s/it]


─── Example 2 ────────────────────────────────────────
  OUTPUT : Step 1 (numbers): math=11, biology=3
Step 2 (operation): difference = 11 - 3 = 8
Answer: 8
  PRED   : 8.0  |  GOLD: 8.0  |  CORRECT: True


SVAMP Structured CoT:   3%|▎         | 3/100 [00:06<03:28,  2.15s/it]

...


─── Example 4 ────────────────────────────────────────
  OUTPUT : Step 1 (numbers): start=4, total=8
Step 2 (operation): added = 8 - 4 = 4
Answer: 4
  PRED   : 4.0  |  GOLD: 4.0  |  CORRECT: True


SVAMP Structured CoT:   5%|▌         | 5/100 [00:10<03:13,  2.03s/it]


─── Example 5 ────────────────────────────────────────
  OUTPUT : Step 1 (numbers): pages=66, days=12, total_days=492
Step 2 (operation): books = 492 / 12 = 41
Answer: 41
  PRED   : 41.0  |  GOLD: 41.0  |  CORRECT: True


SVAMP Structured CoT: 100%|██████████| 100/100 [04:11<00:00,  2.51s/it]


[SVAMP Structured CoT] Accuracy: 87.0% | Null: 0/100


In [20]:
# ── Structured CoT: WinoGrande ───────────────────────────────────────────────
WINO_STRUCT_EXAMPLES = [
    {
        'user': 'Fill the blank using structured reasoning:\nSentence: The trophy doesn\'t fit in the suitcase because _ is too large.\nOption 1: trophy  Option 2: suitcase\nEnd with "Answer: <word>"',
        'assistant': 'Step 1 (referent): The blank is the cause of not fitting.\nStep 2 (reason): What is too large = trophy (it cannot fit inside).\nStep 3 (choose): trophy\nAnswer: trophy'
    },
    {
        'user': 'Fill the blank using structured reasoning:\nSentence: Sam tried to paint a picture of sheep, but they came out looking more like _.\nOption 1: dogs  Option 2: sheep\nEnd with "Answer: <word>"',
        'assistant': 'Step 1 (referent): What the painted result accidentally looks like.\nStep 2 (reason): Sam failed = result looks like something other than sheep = dogs.\nStep 3 (choose): dogs\nAnswer: dogs'
    },
]

def wino_structured_cot_prompt(sample):
    messages = [{'role': 'system', 'content': WINO_SYS}]
    for ex in WINO_STRUCT_EXAMPLES:
        messages.append({'role': 'user',      'content': ex['user']})
        messages.append({'role': 'assistant', 'content': ex['assistant']})
    messages.append({
        'role': 'user',
        'content': (
            f'Fill the blank using structured reasoning:\n'
            f'Sentence: {sample["sentence"]}\n'
            f'Option 1: {sample["option1"]}  Option 2: {sample["option2"]}\n'
            f'End with "Answer: <word>"'
        )
    })
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


acc, wino_scot_records = evaluate(
    wino_samples,
    prompt_fn=wino_structured_cot_prompt,
    extract_fn=wino_extract,
    correct_fn=wino_correct,
    gold_fn=wino_gold,
    label='WinoGrande Structured CoT',
    system_message=None,
)
RESULTS['WinoGrande']['structured_cot'] = acc

WinoGrande Structured CoT:   1%|          | 1/100 [00:02<03:56,  2.39s/it]


─── Example 1 ────────────────────────────────────────
  OUTPUT : Step 1 (referent): The subject who just inherited money.
Step 2 (reason): The one who decided to buy dinner = the one who inherited money.
Step 3 (choose): Felicia
Answer: Felicia
  PRED   : 2  |  GOLD: 1  |  CORRECT: False


WinoGrande Structured CoT:   2%|▏         | 2/100 [00:04<03:37,  2.22s/it]


─── Example 2 ────────────────────────────────────────
  OUTPUT : Step 1 (referent): The location with more snow.
Step 2 (reason): More snow = colder climate = warmer clothing needed.
Step 3 (choose): north
Answer: north
  PRED   : 2  |  GOLD: 2  |  CORRECT: True


WinoGrande Structured CoT:   3%|▎         | 3/100 [00:07<04:08,  2.56s/it]

...


─── Example 4 ────────────────────────────────────────
  OUTPUT : Step 1 (referent): The subject that had more money to spend.
Step 2 (reason): The diner had an easier time following the budget, implying they had more money to spend.
Step 3 (choose): diner
Answer: d
  PRED   : 1  |  GOLD: 1  |  CORRECT: True


WinoGrande Structured CoT:   5%|▌         | 5/100 [00:12<04:09,  2.63s/it]


─── Example 5 ────────────────────────────────────────
  OUTPUT : Step 1 (referent): Identify the cause of not hearing the alarm.
Step 2 (reason): The headphone is the source of sound, and it is closer to John's ears, blocking the alarm sound.
Step 3 (choose): headp
  PRED   : 1  |  GOLD: 1  |  CORRECT: True


WinoGrande Structured CoT: 100%|██████████| 100/100 [04:14<00:00,  2.54s/it]


[WinoGrande Structured CoT] Accuracy: 69.0% | Null: 0/100


## 🔄 Phase 3 — Self-Consistency (5-sample majority vote)

In [21]:
# ── FIX #11: Pick best prompt AFTER Phase 1 results are in ───────────────────
best_svamp_key  = max(RESULTS['SVAMP'],    key=RESULTS['SVAMP'].get)
best_wino_key   = max(RESULTS['WinoGrande'], key=RESULTS['WinoGrande'].get)

svamp_prompt_map = {
    'zero_shot':     (svamp_zeroshot_prompt,       SVAMP_SYS),
    'few_shot':      (svamp_fewshot_prompt,         None),
    'cot':           (svamp_cot_prompt,             SVAMP_SYS),
    'structured_cot':(svamp_structured_cot_prompt,  None),
}
wino_prompt_map = {
    'zero_shot':     (wino_zeroshot_prompt,         WINO_SYS),
    'few_shot':      (wino_fewshot_prompt,           None),
    'cot':           (wino_cot_prompt,              WINO_SYS),
    'structured_cot':(wino_structured_cot_prompt,   None),
}

best_svamp_prompt_fn, best_svamp_sys = svamp_prompt_map[best_svamp_key]
best_wino_prompt_fn,  best_wino_sys  = wino_prompt_map[best_wino_key]

print(f'Best SVAMP prompt    : {best_svamp_key} ({RESULTS["SVAMP"][best_svamp_key]:.1%})')
print(f'Best WinoGrande prompt: {best_wino_key} ({RESULTS["WinoGrande"][best_wino_key]:.1%})')

Best SVAMP prompt    : zero_shot (90.0%)
Best WinoGrande prompt: cot (70.0%)


In [22]:
# ── Self-Consistency: SVAMP ──────────────────────────────────────────────────
correct = 0
null_count = 0

for i, sample in enumerate(tqdm(svamp_samples, desc='SVAMP Self-Consistency')):
    prompt = best_svamp_prompt_fn(sample)
    outputs = generate_n(prompt, n=5, temperature=0.7, system_message=best_svamp_sys)
    answers = [extract_svamp_answer(o) for o in outputs]
    valid   = [round(a, 2) for a in answers if a is not None]

    if not valid:
        null_count += 1
        continue

    pred = Counter(valid).most_common(1)[0][0]
    gold = svamp_gold(sample)
    if svamp_correct(pred, gold):
        correct += 1

    if i < 3:
        print(f'Example {i+1}: preds={answers} → majority={pred}, gold={gold}')

acc = correct / len(svamp_samples)
print(f'\nSVAMP Self-Consistency: {acc:.1%} | Null: {null_count}')
RESULTS['SVAMP']['self_consistency'] = acc

SVAMP Self-Consistency:   1%|          | 1/100 [00:38<1:03:58, 38.77s/it]

Example 1: preds=[21.0, 21.0, 21.0, 21.0, 21.0] → majority=21.0, gold=21.0


SVAMP Self-Consistency:   2%|▏         | 2/100 [00:59<46:01, 28.18s/it]  

Example 2: preds=[8.0, 8.0, 8.0, 8.0, 8.0] → majority=8.0, gold=8.0


SVAMP Self-Consistency:   3%|▎         | 3/100 [01:41<55:58, 34.62s/it]

Example 3: preds=[4.0, 6.0, 6.0, 0.0, 6.0] → majority=6.0, gold=6.0


SVAMP Self-Consistency: 100%|██████████| 100/100 [49:14<00:00, 29.55s/it]


SVAMP Self-Consistency: 90.0% | Null: 0


In [23]:
# ── Self-Consistency: WinoGrande ─────────────────────────────────────────────
correct = 0
null_count = 0

for i, sample in enumerate(tqdm(wino_samples, desc='WinoGrande Self-Consistency')):
    prompt  = best_wino_prompt_fn(sample)
    outputs = generate_n(prompt, n=5, temperature=0.7, system_message=best_wino_sys)
    answers = [extract_wino_answer(o, sample['option1'], sample['option2'])
               for o in outputs]
    valid   = [a for a in answers if a is not None]

    if not valid:
        null_count += 1
        continue

    pred = Counter(valid).most_common(1)[0][0]
    gold = wino_gold(sample)
    if wino_correct(pred, gold):
        correct += 1

    if i < 3:
        print(f'Example {i+1}: preds={answers} → majority={pred}, gold={gold}')

acc = correct / len(wino_samples)
print(f'\nWinoGrande Self-Consistency: {acc:.1%} | Null: {null_count} | Random=50%')
RESULTS['WinoGrande']['self_consistency'] = acc

WinoGrande Self-Consistency:   1%|          | 1/100 [00:14<23:43, 14.38s/it]

Example 1: preds=[1, 2, 2, 2, 2] → majority=2, gold=1


WinoGrande Self-Consistency:   2%|▏         | 2/100 [00:28<23:00, 14.08s/it]

Example 2: preds=[2, 2, 2, 2, 2] → majority=2, gold=2


WinoGrande Self-Consistency:   3%|▎         | 3/100 [00:45<24:57, 15.44s/it]

Example 3: preds=[1, 1, 1, 1, 1] → majority=1, gold=1


WinoGrande Self-Consistency: 100%|██████████| 100/100 [24:11<00:00, 14.51s/it]


WinoGrande Self-Consistency: 71.0% | Null: 0 | Random=50%


## ✅ Phase 4 — Rule-Based VCE

In [24]:
# ── Rule-based VCE: SVAMP arithmetic verifier ────────────────────────────────

def arithmetic_verify(output: str) -> bool:
    """
    Find arithmetic expressions like 'X op Y = Z' and verify them.
    Returns True  → no errors found (passes).
    Returns False → at least one verifiable arithmetic error detected.
    """
    pattern = r'(-?\d+\.?\d*)\s*([+\-×x*/÷])\s*(-?\d+\.?\d*)\s*=\s*(-?\d+\.?\d*)'
    ops = re.findall(pattern, output)

    if not ops:
        return True  # Nothing to verify — let it pass

    op_map = {
        '+': lambda a, b: a + b,
        '-': lambda a, b: a - b,
        '*': lambda a, b: a * b,
        'x': lambda a, b: a * b,
        '×': lambda a, b: a * b,
        '/': lambda a, b: a / b if b != 0 else None,
        '÷': lambda a, b: a / b if b != 0 else None,
    }

    for a_s, op, b_s, r_s in ops:
        try:
            a, b, claimed = float(a_s), float(b_s), float(r_s)
            fn = op_map.get(op)
            if fn:
                actual = fn(a, b)
                if actual is not None and abs(actual - claimed) > 0.05:
                    return False
        except:
            continue
    return True


correct = 0
vce_overrides = 0
null_count = 0

for i, sample in enumerate(tqdm(svamp_samples, desc='SVAMP SC + VCE')):
    prompt  = best_svamp_prompt_fn(sample)
    outputs = generate_n(prompt, n=5, temperature=0.7, system_message=best_svamp_sys)
    candidates = [(extract_svamp_answer(o), o) for o in outputs]
    valid = [(ans, out) for ans, out in candidates if ans is not None]

    if not valid:
        null_count += 1
        continue

    sc_answer  = Counter([round(a, 2) for a, _ in valid]).most_common(1)[0][0]
    verified   = [(ans, out) for ans, out in valid if arithmetic_verify(out)]
    vce_answer = Counter([round(a, 2) for a, _ in verified]).most_common(1)[0][0] \
        if verified else sc_answer

    if vce_answer != sc_answer:
        vce_overrides += 1

    if svamp_correct(vce_answer, svamp_gold(sample)):
        correct += 1

acc = correct / len(svamp_samples)
print(f'SVAMP SC + VCE: {acc:.1%} | VCE overrides: {vce_overrides} | Null: {null_count}')
RESULTS['SVAMP']['sc_vce'] = acc

SVAMP SC + VCE: 100%|██████████| 100/100 [49:37<00:00, 29.78s/it]

SVAMP SC + VCE: 96.0% | VCE overrides: 0 | Null: 0


In [25]:
# ── Rule-based VCE: WinoGrande consistency checker ───────────────────────────

def wino_verify(output: str, option1: str, option2: str, pred: int) -> bool:
    """
    Returns True if reasoning is consistent with the chosen option.
    Returns False if reasoning explicitly concludes the OTHER option.
    """
    if pred is None:
        return False

    chosen   = (option1 if pred == 1 else option2).lower()
    rejected = (option2 if pred == 1 else option1).lower()
    text = output.lower()

    contradiction_patterns = [
        rf'therefore\s+{re.escape(rejected)}',
        rf'so\s+{re.escape(rejected)}',
        rf'answer\s+is\s+{re.escape(rejected)}',
        rf'answer:\s+{re.escape(rejected)}',
        rf'choose\s+{re.escape(rejected)}',
        rf'correct.*{re.escape(rejected)}',
    ]
    for p in contradiction_patterns:
        if re.search(p, text):
            return False
    return True


correct = 0
vce_overrides = 0
null_count = 0

for i, sample in enumerate(tqdm(wino_samples, desc='WinoGrande SC + VCE')):
    prompt  = best_wino_prompt_fn(sample)
    outputs = generate_n(prompt, n=5, temperature=0.7, system_message=best_wino_sys)
    o1, o2  = sample['option1'], sample['option2']
    candidates = [(extract_wino_answer(out, o1, o2), out) for out in outputs]
    valid = [(ans, out) for ans, out in candidates if ans is not None]

    if not valid:
        null_count += 1
        continue

    sc_answer  = Counter([a for a, _ in valid]).most_common(1)[0][0]
    verified   = [(ans, out) for ans, out in valid if wino_verify(out, o1, o2, ans)]
    vce_answer = Counter([a for a, _ in verified]).most_common(1)[0][0] \
        if verified else sc_answer

    if vce_answer != sc_answer:
        vce_overrides += 1

    if wino_correct(vce_answer, wino_gold(sample)):
        correct += 1

acc = correct / len(wino_samples)
print(f'WinoGrande SC + VCE: {acc:.1%} | VCE overrides: {vce_overrides} | Null: {null_count}')
RESULTS['WinoGrande']['sc_vce'] = acc

WinoGrande SC + VCE: 100%|██████████| 100/100 [24:33<00:00, 14.74s/it]

WinoGrande SC + VCE: 65.0% | VCE overrides: 0 | Null: 0


## 📊 Phase 5 — Final Results

In [26]:
PIPELINE_LABELS = {
    'zero_shot':        'Zero-Shot (baseline)',
    'few_shot':         'Few-Shot (baseline)',
    'cot':              'Chain-of-Thought (baseline)',
    'structured_cot':   '+ Structured CoT     [Phase 2]',
    'self_consistency': '+ Self-Consistency 5x [Phase 3]',
    'sc_vce':           '+ SC + Rule-VCE       [Phase 4]',
}

print('═' * 68)
print(f'{"Component":<40} {"SVAMP":>10} {"WinoGrande":>12}')
print('─' * 68)
for key, label in PIPELINE_LABELS.items():
    sv = RESULTS['SVAMP'].get(key)
    wi = RESULTS['WinoGrande'].get(key)
    s  = f'{sv:.1%}' if sv is not None else 'N/A'
    w  = f'{wi:.1%}' if wi is not None else 'N/A'
    print(f'{label:<40} {s:>10} {w:>12}')
print('═' * 68)
print(f'{"Random baseline (WinoGrande)":40} {"N/A":>10} {"50.0%":>12}')
print('═' * 68)

print()
for bm in ['SVAMP', 'WinoGrande']:
    baselines = {k: v for k, v in RESULTS[bm].items()
                 if k in ['zero_shot', 'few_shot', 'cot']}
    pipeline  = {k: v for k, v in RESULTS[bm].items()
                 if k not in ['zero_shot', 'few_shot', 'cot']}
    if baselines and pipeline:
        best_bl = max(baselines.values())
        best_pl = max(pipeline.values())
        delta   = best_pl - best_bl
        sign    = '+' if delta >= 0 else ''
        verdict = '✅ PIPELINE WINS' if delta > 0 else '❌ NO IMPROVEMENT'
        print(f'{bm}: best_baseline={best_bl:.1%}  best_pipeline={best_pl:.1%}  '
              f'Δ={sign}{delta:.1%}  {verdict}')

# Save results
output_path = '/kaggle/working/sldvm_results.json'
try:
    with open(output_path, 'w') as f:
        json.dump(RESULTS, f, indent=2)
    print(f'\n✅ Results saved to {output_path}')
except Exception as e:
    print(f'\nNote: Could not save to {output_path}: {e}')
    # Try alternate path
    with open('sldvm_results.json', 'w') as f:
        json.dump(RESULTS, f, indent=2)
    print('✅ Results saved to sldvm_results.json')

════════════════════════════════════════════════════════════════════
Component                                     SVAMP   WinoGrande
────────────────────────────────────────────────────────────────────
Zero-Shot (baseline)                          90.0%        68.0%
Few-Shot (baseline)                           90.0%        67.0%
Chain-of-Thought (baseline)                   89.0%        70.0%
+ Structured CoT     [Phase 2]                87.0%        69.0%
+ Self-Consistency 5x [Phase 3]               90.0%        71.0%
+ SC + Rule-VCE       [Phase 4]               96.0%        65.0%
════════════════════════════════════════════════════════════════════
Random baseline (WinoGrande)                    N/A        50.0%
════════════════════════════════════════════════════════════════════

SVAMP: best_baseline=90.0%  best_pipeline=96.0%  Δ=+6.0%  ✅ PIPELINE WINS
WinoGrande: best_baseline=70.0%  best_pipeline=71.0%  Δ=+1.0%  ✅ PIPELINE WINS

✅ Results saved to /kaggle/working/sldvm_results.

In [27]:
# ── Failure analysis ─────────────────────────────────────────────────────────

print('FAILURE ANALYSIS — SVAMP (first 10 wrong, zero-shot baseline)')
print('─' * 60)
wrong = [r for r in svamp_zs_records if not r['correct']]
for i, r in enumerate(wrong[:10]):
    bq = svamp_body_q(r['sample'])
    print(f'\n[{i+1}] Problem: {bq[:120]}')
    print(f'     Gold: {r["gold"]}  Pred: {r["pred"]}')
    print(f'     Output: {r["output"][:250]}')

print('\n')
print('FAILURE ANALYSIS — WinoGrande (first 10 wrong, zero-shot baseline)')
print('─' * 60)
wrong = [r for r in wino_zs_records if not r['correct']]
for i, r in enumerate(wrong[:10]):
    s = r['sample']
    print(f'\n[{i+1}] Sentence: {s["sentence"]}')
    print(f'     Opt1={s["option1"]}  Opt2={s["option2"]}  Gold={r["gold"]}  Pred={r["pred"]}')
    print(f'     Output: {r["output"][:200]}')

FAILURE ANALYSIS — SVAMP (first 10 wrong, zero-shot baseline)
────────────────────────────────────────────────────────────

[1] Problem: There are 17 different movies and 11 different books in the ' crazy silly school ' series. If you read 13 of the books a
     Gold: 6.0  Pred: 0.0
     Output: First, let's find out how many movies and books are left after reading 13 books and watching 63 movies.

There are 17 different movies in total, and you watched 63 of them. Since it's not possible to watch more movies than there are in the series, th

[2] Problem: Bobby had 20 pieces of candy. He ate 34 pieces of candy. Then he ate 18 more. How many pieces of candy did Bobby eat?
     Gold: 52.0  Pred: 20.0
     Output: Step 1: Bobby started with 20 pieces of candy.
Step 2: He ate 34 pieces of candy, but since he only had 20 pieces, this step is not possible. He cannot eat more candy than he has.
Step 3: The problem states that he ate 18 more pieces of candy, but ag

[3] Problem: Matthew had 24